# Snæfellsjökull DEM (ÍslandsDEM v1.0, 10 m)

This notebook downloads a **10 m digital elevation model (DEM)** of Snæfellsjökull and its surroundings from Google Earth Engine, and produces:

- the DEM as a GeoTIFF, on its original 10 m grid in the Icelandic national grid ISN93
- a hillshade and a slope map (GeoTIFFs), for maps and terrain analysis
- a preview map with contours and glacier outlines
- elevation statistics for the glacier
- a heightmap for the 3D viewer in `../Web/Model/`

## Data source

| | |
|---|---|
| **Dataset** | ÍslandsDEM v1.0 (*Hæðarlíkan útgáfa 1.0*) |
| **Earth Engine asset** | `projects/ee-landmaelingar/assets/IslandsDEMv1_10m_isn93` |
| **Publisher** | National Land Survey of Iceland (*Landmælingar Íslands*). The national metadata catalogue now lists the Natural Science Institute of Iceland (*Náttúrufræðistofnun*) as the contact. |
| **Resolution** | 10 m |
| **Projection** | ISN93 / Lambert 1993 (EPSG:3057) |
| **Band** | `b1`, elevation in metres (float) |
| **Metadata** | [Catalogue record](https://gatt.lmi.is/geonetwork/srv/eng/catalog.search#/metadata/e6712430-a63c-4ae5-9158-c89d16da6361) |

**How it was made.** ÍslandsDEM is built from **ArcticDEM** (Polar Geospatial Center, University of Minnesota): 2 m DEMs made from WorldView and GeoEye stereo satellite images, mostly acquired between 2012 and about 2020, some as early as 2008. The National Land Survey aligned all strips over Iceland and merged them. Each pixel is the **median** of all the elevations available for that spot. The result was then resampled to 10 m.

**What that means for a glacier.** Because each pixel is a median over several years, the glacier surface **is not a snapshot from one date**. Snæfellsjökull has been thinning, so the ice surface here is a blend of roughly 2008–2020 surfaces. The DEM is good for terrain context, maps, hillshades and approximate elevations. **Don't use it to measure thinning**; for that you need dated DEMs (e.g. individual ArcticDEM strips or the 2008 lidar survey).

**Vertical datum.** The catalogue doesn't state it. The highest point in the DEM here is 1,438 m, close to the published summit height of 1,446 m. That suggests heights above sea level rather than above the ellipsoid (raw ArcticDEM heights are above the WGS84 ellipsoid, which is tens of metres higher in Iceland). Check with the publisher before combining it with other elevation data.

**Use and credit.** The asset is public in Earth Engine. Credit *Landmælingar Íslands (National Land Survey of Iceland)* and ArcticDEM. The catalogue doesn't state a license, so check the terms before publishing. The same DEM can also be downloaded without Earth Engine from the [DEM map viewer](https://kort.gis.is/mapview/?app=DEM) or the publisher's [file server](https://ftp.natt.is/gisdata/raster/).

**Glacier outlines** for the download area and the preview map come from the Icelandic Meteorological Office (Veðurstofa Íslands), CC-BY 4.0. They are the same source as `../Snæfellsjökull_outlines/download_outlines.ipynb`.

## Outputs

Rasters are written to `outputs/` next to this notebook, in ISN93 (EPSG:3057) on the DEM's own 10 m grid, so they line up pixel for pixel with the full ÍslandsDEM.

| Path | Contents |
|---|---|
| `outputs/snaefellsjokull_dem_10m_isn93.tif` | Elevation in metres, float32. No-data value `-9999`. |
| `outputs/snaefellsjokull_hillshade_10m_isn93.tif` | Hillshade, 0–255 (uint8). Light from the northwest (azimuth 315°, altitude 45°). |
| `outputs/snaefellsjokull_slope_10m_isn93.tif` | Slope in degrees, float32 |
| `outputs/snaefellsjokull_dem_10m_isn93_wide.tif` | A wider copy of the DEM (~1890 outline + 5 km), used only by `cross_section.ipynb` (section 10) |
| `outputs/snaefellsjokull_dem_preview.png` | Map of the hillshade with 100 m contours and glacier outlines |
| `outputs/snaefellsjokull_glacier_elevation_stats.csv` | Elevation statistics inside the newest glacier outline |
| `outputs/README.md` | Short description of the files, with attribution |
| `../Web/Model/data/snaefellsjokull_dem.bin` and `.json` | Heightmap and its description, for the 3D viewer (section 9) |
| `../Web/Model/data/snaefellsjokull_outlines.json` | Glacier outlines of every mapped year, in ISN93, for the 3D viewer (section 9) |

### Figure style

All figures use one minimal black-and-white style: white background, black text and lines, greys for secondary elements, no gridlines and no top or right axis borders. Satellite images keep their natural colours, because the colour is the data.

## 1. Setup

In [ ]:
# Install dependencies (run once)
%pip install -q earthengine-api requests rasterio geopandas matplotlib numpy pandas

In [ ]:
import json
from pathlib import Path

import ee
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import requests
from matplotlib.colors import LightSource
from rasterio.features import geometry_mask
from shapely.geometry import box

In [ ]:
from cycler import cycler

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "text.color": "black",
    "axes.edgecolor": "black",
    "axes.labelcolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
    "axes.prop_cycle": cycler(color=["black"]),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": False,
    "axes.linewidth": 0.8,
    "axes.titlesize": 11,
    "axes.titlelocation": "left",
    "font.family": "sans-serif",
    "font.size": 10,
    "lines.linewidth": 1.2,
    "legend.frameon": False,
})

GREY = "#9a9a9a"
LIGHT_GREY = "#d4d4d4"

In [ ]:
# Authenticate and initialize Earth Engine.
# Use a Google Cloud project that has the Earth Engine API enabled.
PROJECT = "gsapp-map"

try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT)

print("Earth Engine initialized")

## 2. Settings

The download area is the **~1890 (Little Ice Age) outline**, the largest the glacier has been in recorded history, plus a margin of `BUFFER_M` metres on every side. That covers all of the ice cap's mapped positions and the moraines in front of it. Increase `BUFFER_M` for more surrounding terrain.

If the outline server can't be reached, `FALLBACK_BBOX` (longitude/latitude) is used instead.

In [ ]:
DEM_ASSET = "projects/ee-landmaelingar/assets/IslandsDEMv1_10m_isn93"
DEM_BAND = "b1"
DEM_CRS = "EPSG:3057"  # ISN93 / Lambert 1993, the DEM's native projection

GLACIER_NAME = "Snæfellsjökull"
GLACIER_ID = "G336220E64811N"  # GLIMS ID
SLUG = "snaefellsjokull"

# Glacier outlines (Icelandic Met Office)
WFS_URL = "https://geo.vedur.is/geoserver/wfs"
EXTENT_LAYER = "glaciology:glacierpolygon_lia"         # ~1890 outline, sets the download area
LATEST_LAYER = "glaciology:glacierpolygon_sent2_2025"  # newest outline, for statistics and the map

BUFFER_M = 1000                                 # margin around the ~1890 outline
FALLBACK_BBOX = [-23.95, 64.74, -23.60, 64.88]  # lon/lat: west, south, east, north

NODATA = -9999.0
CONTOUR_INTERVAL_M = 100

OUT_DIR = Path("outputs")
WEB_DATA_DIR = Path("../Web/Model/data")  # heightmap for the 3D viewer
OUT_DIR.mkdir(parents=True, exist_ok=True)

DEM_PATH = OUT_DIR / f"{SLUG}_dem_10m_isn93.tif"
HILLSHADE_PATH = OUT_DIR / f"{SLUG}_hillshade_10m_isn93.tif"
SLOPE_PATH = OUT_DIR / f"{SLUG}_slope_10m_isn93.tif"

## 3. Inspect the DEM asset

In [ ]:
dem = ee.Image(DEM_ASSET).select(DEM_BAND)

proj = dem.projection().getInfo()
NATIVE_TRANSFORM = proj["transform"]  # [x scale, x shear, x origin, y shear, y scale, y origin]
PIXEL = NATIVE_TRANSFORM[0]

print("CRS:      ", proj["crs"])
print("Transform:", NATIVE_TRANSFORM)
print("Pixel:    ", PIXEL, "m")
assert proj["crs"] == DEM_CRS, "The asset's projection has changed; update DEM_CRS"

## 4. Download area

The ~1890 outline is downloaded in ISN93, its bounding box is expanded by `BUFFER_M`, and the box edges are snapped to the DEM's 10 m pixel grid. Snapping means Earth Engine returns the original pixels exactly, with no resampling.

In [ ]:
def fetch_outline(layer):
    """Download this glacier's outline from one Met Office layer, merged into one polygon, in DEM_CRS."""
    params = {
        "service": "WFS",
        "version": "2.0.0",
        "request": "GetFeature",
        "typeNames": layer,
        "outputFormat": "application/json",
        "srsName": DEM_CRS,
        "CQL_FILTER": f"id='{GLACIER_ID}'",
    }
    r = requests.get(WFS_URL, params=params, timeout=120)
    r.raise_for_status()
    gdf = gpd.GeoDataFrame.from_features(r.json()["features"], crs=DEM_CRS)
    return gdf.dissolve()[["geometry"]]


try:
    extent_outline = fetch_outline(EXTENT_LAYER)
    latest_outline = fetch_outline(LATEST_LAYER)
    xmin, ymin, xmax, ymax = extent_outline.total_bounds
    xmin, ymin, xmax, ymax = xmin - BUFFER_M, ymin - BUFFER_M, xmax + BUFFER_M, ymax + BUFFER_M
    print("Download area: ~1890 outline +", BUFFER_M, "m")
except Exception as e:
    print("Outline server unavailable, using FALLBACK_BBOX:", e)
    extent_outline = latest_outline = None
    fallback = gpd.GeoSeries([box(*FALLBACK_BBOX)], crs="EPSG:4326").to_crs(DEM_CRS)
    xmin, ymin, xmax, ymax = fallback.total_bounds

# Snap outward to the DEM's pixel grid
x0, y0 = NATIVE_TRANSFORM[2], NATIVE_TRANSFORM[5]
xmin = x0 + np.floor((xmin - x0) / PIXEL) * PIXEL
xmax = x0 + np.ceil((xmax - x0) / PIXEL) * PIXEL
ymin = y0 - np.ceil((y0 - ymin) / PIXEL) * PIXEL
ymax = y0 - np.floor((y0 - ymax) / PIXEL) * PIXEL

width = int(round((xmax - xmin) / PIXEL))
height = int(round((ymax - ymin) / PIXEL))
download_transform = [PIXEL, 0, float(xmin), 0, -PIXEL, float(ymax)]

print(f"Bounds (ISN93): x {xmin:.0f}–{xmax:.0f}, y {ymin:.0f}–{ymax:.0f}")
print(f"Size: {width} x {height} pixels = {width * PIXEL / 1000:.1f} x {height * PIXEL / 1000:.1f} km")
print(f"Approximate file size: {width * height * 4 / 1e6:.1f} MB")

## 5. Download the DEM

`getDownloadURL` downloads the GeoTIFF directly. It's limited to about **48 MB** per request (about 3,400 × 3,400 pixels at 10 m). The default area is far smaller. For a much larger area, use the Google Drive export in section 10 instead.

Pixels outside the DEM's coverage are filled with `NODATA` (-9999) and tagged as no-data in the file. There are none in the default area.

In [ ]:
if width * height * 4 > 48e6:
    raise ValueError("Area too large for a direct download; reduce BUFFER_M or use the Drive export below")

dem_download = dem.unmask(NODATA, False).toFloat().rename("elevation")
url = dem_download.getDownloadURL({
    "format": "GEO_TIFF",
    "crs": DEM_CRS,
    "crs_transform": download_transform,
    "dimensions": f"{width}x{height}",
})

response = requests.get(url, timeout=600)
response.raise_for_status()
DEM_PATH.write_bytes(response.content)

# Record the no-data value and the source in the GeoTIFF itself
with rasterio.open(DEM_PATH, "r+") as ds:
    ds.nodata = NODATA
    ds.update_tags(
        source="ÍslandsDEM v1.0, Landmælingar Íslands (National Land Survey of Iceland), via Google Earth Engine",
        ee_asset=DEM_ASSET,
        units="metres",
    )
    ds.set_band_description(1, "elevation")

with rasterio.open(DEM_PATH) as src:
    print(f"wrote {DEM_PATH}: {src.width} x {src.height} px, {src.res[0]} m, {src.crs}")
    print("Bounds:", tuple(round(v) for v in src.bounds))

## 6. Hillshade and slope

These are calculated locally from the downloaded DEM with matplotlib's `LightSource`. Pixels at the edge of the area have one-sided slopes, so they are slightly less accurate.

In [ ]:
with rasterio.open(DEM_PATH) as src:
    elevation = src.read(1, masked=True)
    profile = src.profile
    bounds = src.bounds
    transform = src.transform

z = elevation.filled(np.nan).astype("float64")

hillshade = LightSource(azdeg=315, altdeg=45).hillshade(z, dx=PIXEL, dy=PIXEL)
hillshade_u8 = np.where(np.isnan(z), 0, np.round(hillshade * 255)).astype("uint8")

dz_dy, dz_dx = np.gradient(z, PIXEL)
slope = np.degrees(np.arctan(np.hypot(dz_dx, dz_dy))).astype("float32")
slope = np.where(np.isnan(z), NODATA, slope).astype("float32")

with rasterio.open(HILLSHADE_PATH, "w", **{**profile, "dtype": "uint8", "nodata": 0}) as dst:
    dst.write(hillshade_u8, 1)
    dst.set_band_description(1, "hillshade (azimuth 315, altitude 45)")

with rasterio.open(SLOPE_PATH, "w", **{**profile, "dtype": "float32", "nodata": NODATA}) as dst:
    dst.write(slope, 1)
    dst.set_band_description(1, "slope (degrees)")

print("wrote", HILLSHADE_PATH)
print("wrote", SLOPE_PATH)

## 7. Elevation statistics

The highest point in the whole area, plus elevation statistics for the ice inside the newest glacier outline. The summit crags are bare rock outside the outline, so the ice's highest point is lower than the area's.

In [ ]:
# Highest point in the area
row, col = np.unravel_index(np.nanargmax(z), z.shape)
peak_x, peak_y = rasterio.transform.xy(transform, row, col)
peak = gpd.GeoSeries(gpd.points_from_xy([peak_x], [peak_y]), crs=DEM_CRS).to_crs("EPSG:4326").iloc[0]
print(f"Highest point: {np.nanmax(z):.0f} m at {peak.y:.5f}° N, {abs(peak.x):.5f}° W")
print(f"Lowest point in the area: {np.nanmin(z):.0f} m")

stats = None
if latest_outline is not None:
    inside = ~geometry_mask(latest_outline.geometry, out_shape=z.shape, transform=transform)
    ice = z[inside & ~np.isnan(z)]
    stats = pd.DataFrame([{
        "outline_layer": LATEST_LAYER,
        "glacier_area_km2": round(inside.sum() * PIXEL * PIXEL / 1e6, 2),
        "min_elev_m": round(float(ice.min()), 1),
        "p10_elev_m": round(float(np.percentile(ice, 10)), 1),
        "median_elev_m": round(float(np.median(ice)), 1),
        "mean_elev_m": round(float(ice.mean()), 1),
        "p90_elev_m": round(float(np.percentile(ice, 90)), 1),
        "max_elev_m": round(float(ice.max()), 1),
    }])
    stats.to_csv(OUT_DIR / f"{SLUG}_glacier_elevation_stats.csv", index=False)
    display(stats.T.rename(columns={0: GLACIER_NAME}))

## 8. Preview map

Greyscale hillshade with 100 m contours (labelled every 500 m), the ~1890 outline (dashed) and the newest outline (solid).

In [ ]:
extent = (bounds.left, bounds.right, bounds.bottom, bounds.top)

fig, ax = plt.subplots(figsize=(9, 9.5))
ax.imshow(hillshade, cmap="gray", extent=extent, vmin=-0.15, vmax=1.15)

levels = np.arange(0, np.nanmax(z) + CONTOUR_INTERVAL_M, CONTOUR_INTERVAL_M)
xs = np.linspace(bounds.left + PIXEL / 2, bounds.right - PIXEL / 2, z.shape[1])
ys = np.linspace(bounds.top - PIXEL / 2, bounds.bottom + PIXEL / 2, z.shape[0])
contours = ax.contour(xs, ys, z, levels=levels, colors="black", linewidths=0.35, alpha=0.6)
labelled = [lvl for lvl in levels if lvl % 500 == 0 and lvl > 0]
ax.clabel(contours, levels=labelled, fontsize=7, fmt="%d m")

if extent_outline is not None:
    extent_outline.boundary.plot(ax=ax, color="black", linewidth=1.0, linestyle=(0, (4, 3)))
    latest_outline.boundary.plot(ax=ax, color="black", linewidth=1.6)
    ax.plot([], [], color="black", linewidth=1.0, linestyle=(0, (4, 3)), label="~1890 outline")
    ax.plot([], [], color="black", linewidth=1.6, label=LATEST_LAYER.split("_")[-1] + " outline")
    ax.legend(loc="upper left", fontsize=8, frameon=True, facecolor="white", edgecolor="none", framealpha=0.9)

ax.plot(peak_x, peak_y, marker="^", color="black", markersize=7, linestyle="none")
ax.annotate(f"{np.nanmax(z):.0f} m", (peak_x, peak_y), textcoords="offset points", xytext=(7, 4), fontsize=8)

# 1 km scale bar
sb_x = bounds.right - 0.04 * (bounds.right - bounds.left) - 1000
sb_y = bounds.bottom + 0.03 * (bounds.top - bounds.bottom)
ax.plot([sb_x, sb_x + 1000], [sb_y, sb_y], color="black", linewidth=2.5, solid_capstyle="butt")
ax.text(sb_x + 500, sb_y + 80, "1 km", ha="center", va="bottom", fontsize=8)

ax.set_xlim(bounds.left, bounds.right)
ax.set_ylim(bounds.bottom, bounds.top)
ax.axis("off")
ax.set_title(f"{GLACIER_NAME} · ÍslandsDEM v1.0, 10 m · contours every {CONTOUR_INTERVAL_M} m")
ax.text(0, -0.01, "DEM: Landmælingar Íslands / ArcticDEM · Outlines: Veðurstofa Íslands, CC-BY 4.0",
        transform=ax.transAxes, va="top", fontsize=7, color=GREY)
fig.tight_layout()
fig.savefig(OUT_DIR / f"{SLUG}_dem_preview.png", dpi=150, bbox_inches="tight", pad_inches=0.2)
plt.show()

## 9. Heightmap for the 3D web viewer

The viewer in `../Web/Model/` reads three files from `../Web/Model/data/`:

- `snaefellsjokull_dem.bin`: every elevation as a 32-bit little-endian float, row by row from the **north** edge, west to east within a row. No-data pixels are `-9999`.
- `snaefellsjokull_dem.json`: the grid size, pixel size, elevation range, bounds and credits the viewer needs to read the `.bin` file.
- `snaefellsjokull_outlines.json`: every mapped glacier outline (~1890 to 2025) in ISN93 metres, drawn over the terrain. It's made from the GeoPackage written by `../Snæfellsjökull_outlines/download_outlines.ipynb`, so run that notebook first.

The full 10 m grid is written (about 2.6 MB). The viewer can draw it at lower detail.

In [ ]:
WEB_DATA_DIR.mkdir(parents=True, exist_ok=True)

heights = np.where(np.isnan(z), NODATA, z).astype("<f4")  # little-endian float32, row 0 = north
(WEB_DATA_DIR / f"{SLUG}_dem.bin").write_bytes(heights.tobytes())

web_meta = {
    "name": GLACIER_NAME,
    "width": int(z.shape[1]),
    "height": int(z.shape[0]),
    "pixel_m": float(PIXEL),
    "min_m": round(float(np.nanmin(z)), 2),
    "max_m": round(float(np.nanmax(z)), 2),
    "nodata": NODATA,
    "crs": DEM_CRS,
    "bounds": {"left": bounds.left, "bottom": bounds.bottom, "right": bounds.right, "top": bounds.top},
    "layout": "float32 little-endian, row-major, first row = north edge, west to east",
    "source": "ÍslandsDEM v1.0 (10 m), Landmælingar Íslands / ArcticDEM",
}
(WEB_DATA_DIR / f"{SLUG}_dem.json").write_text(json.dumps(web_meta, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"wrote {WEB_DATA_DIR / f'{SLUG}_dem.bin'} ({heights.nbytes / 1e6:.1f} MB)")
print(f"wrote {WEB_DATA_DIR / f'{SLUG}_dem.json'}")

In [ ]:
# Glacier outlines for the viewer, in the DEM's coordinates (ISN93 metres)
OUTLINES_GPKG = Path("../Snæfellsjökull_outlines/outputs/snaefellsjokull_outlines_all_years.gpkg")


def outline_rings(geom):
    """Every ring of a (multi)polygon as [[x, y], ...]: outer edges and holes (rock islands in the ice)."""
    rings = []
    for poly in getattr(geom, "geoms", [geom]):
        for ring in [poly.exterior, *poly.interiors]:
            rings.append([[round(x, 1), round(y, 1)] for x, y in ring.coords])
    return rings


def describe_source(source):
    """Plain-English version of the Met Office's source field."""
    s = source.lower()
    for key, text in [
        ("geomorph", "moraines left by the Little Ice Age maximum"),
        ("ams", "US Army Map Service aerial photos"),
        ("nlsi", "National Land Survey aerial photos"),
        ("loftmyndir", "Loftmyndir ehf aerial photos"),
        ("lidar", "airborne lidar survey"),
        ("arcticdem", "ArcticDEM satellite elevation model"),
        ("sentinel", "Sentinel-2 satellite image"),
    ]:
        if key in s:
            return text
    return source


if OUTLINES_GPKG.exists():
    all_outlines = gpd.read_file(OUTLINES_GPKG).to_crs(DEM_CRS).sort_values("year")
    web_outlines = {
        "crs": DEM_CRS,
        "source": "Veðurstofa Íslands (Icelandic Met Office), CC-BY 4.0",
        "outlines": [
            {"year": int(row.year), "label": row.label, "date": row.acq_date,
             "area_km2": round(float(row.area_km2), 2), "source": describe_source(row.source),
             "rings": outline_rings(row.geometry)}
            for row in all_outlines.itertuples()
        ],
    }
    outlines_path = WEB_DATA_DIR / f"{SLUG}_outlines.json"
    outlines_path.write_text(json.dumps(web_outlines, ensure_ascii=False, separators=(",", ":")), encoding="utf-8")
    print(f"wrote {outlines_path} ({len(web_outlines['outlines'])} years, {outlines_path.stat().st_size / 1e3:.0f} kB)")
else:
    print(f"{OUTLINES_GPKG} not found: run ../Snæfellsjökull_outlines/download_outlines.ipynb, then this cell again")

## 10. Wider DEM for the cross-sections

`cross_section.ipynb` draws the rock as the DEM mesh *behind* each section line. Lines near the edge of the ice cap (like the one across the north flank) need terrain further out than the main DEM covers. This downloads a second, wider copy, the ~1890 outline plus `WIDE_BUFFER_M` on every side, on the same 10 m grid. Only the cross-sections use it, so the main DEM and the web viewer stay the same.

The wider area reaches the sea on the south and west. ÍslandsDEM has no values over the sea, so those pixels are `-9999` (no-data), as in the main DEM.

In [ ]:
WIDE_BUFFER_M = 5000
WIDE_DEM_PATH = OUT_DIR / f"{SLUG}_dem_10m_isn93_wide.tif"

if extent_outline is not None:
    wxmin, wymin, wxmax, wymax = extent_outline.total_bounds
else:
    wxmin, wymin, wxmax, wymax = gpd.GeoSeries([box(*FALLBACK_BBOX)], crs="EPSG:4326").to_crs(DEM_CRS).total_bounds
wxmin, wymin, wxmax, wymax = wxmin - WIDE_BUFFER_M, wymin - WIDE_BUFFER_M, wxmax + WIDE_BUFFER_M, wymax + WIDE_BUFFER_M

# Snap outward to the DEM's pixel grid, as for the main DEM
gx, gy = NATIVE_TRANSFORM[2], NATIVE_TRANSFORM[5]
wxmin = gx + np.floor((wxmin - gx) / PIXEL) * PIXEL
wxmax = gx + np.ceil((wxmax - gx) / PIXEL) * PIXEL
wymin = gy - np.ceil((gy - wymin) / PIXEL) * PIXEL
wymax = gy - np.floor((gy - wymax) / PIXEL) * PIXEL
wide_width = int(round((wxmax - wxmin) / PIXEL))
wide_height = int(round((wymax - wymin) / PIXEL))
if wide_width * wide_height * 4 > 48e6:
    raise ValueError("Wide area too large for a direct download; reduce WIDE_BUFFER_M")

url = dem.unmask(NODATA, False).toFloat().rename("elevation").getDownloadURL({
    "format": "GEO_TIFF",
    "crs": DEM_CRS,
    "crs_transform": [PIXEL, 0, float(wxmin), 0, -PIXEL, float(wymax)],
    "dimensions": f"{wide_width}x{wide_height}",
})
response = requests.get(url, timeout=600)
response.raise_for_status()
WIDE_DEM_PATH.write_bytes(response.content)

with rasterio.open(WIDE_DEM_PATH, "r+") as ds:
    ds.nodata = NODATA
    ds.update_tags(
        source="ÍslandsDEM v1.0, Landmælingar Íslands (National Land Survey of Iceland), via Google Earth Engine",
        ee_asset=DEM_ASSET,
        units="metres",
        area=f"~1890 outline + {WIDE_BUFFER_M} m",
    )
    ds.set_band_description(1, "elevation")

print(f"wrote {WIDE_DEM_PATH}: {wide_width} x {wide_height} px "
      f"({wide_width * PIXEL / 1000:.1f} x {wide_height * PIXEL / 1000:.1f} km, {WIDE_DEM_PATH.stat().st_size / 1e6:.0f} MB)")

## 11. (Optional) Export a larger area to Google Drive

For areas larger than the direct-download limit, set `EXPORT_TO_DRIVE = True`. Earth Engine writes the GeoTIFF to a folder in your Google Drive, splitting it into tiles if needed. Track progress at https://code.earthengine.google.com/tasks.

In [ ]:
EXPORT_TO_DRIVE = False
DRIVE_FOLDER = "snaefellsjokull"

if EXPORT_TO_DRIVE:
    task = ee.batch.Export.image.toDrive(
        image=dem.toFloat(),
        description=f"{SLUG}_dem_10m_isn93",
        folder=DRIVE_FOLDER,
        crs=DEM_CRS,
        crsTransform=download_transform,
        dimensions=f"{width}x{height}",
        maxPixels=1e10,
    )
    task.start()
    print("Export started:", task.id)

## 12. README for the output folder

In [ ]:
if stats is not None:
    stats_text = "| Statistic | Value |\n|---|---|\n" + "\n".join(
        f"| {k} | {v} |" for k, v in stats.iloc[0].items()
    )
else:
    stats_text = "(not calculated)"

readme = f"""# {GLACIER_NAME} DEM

10 m digital elevation model of {GLACIER_NAME} and its surroundings, cut from ÍslandsDEM v1.0
(Earth Engine asset `{DEM_ASSET}`) by `download_dem.ipynb`.

**Credit:** Landmælingar Íslands (National Land Survey of Iceland), ÍslandsDEM v1.0, built from ArcticDEM
(Polar Geospatial Center, University of Minnesota). Glacier outlines: Veðurstofa Íslands (Icelandic Met Office), CC-BY 4.0.

## Files

| File | Contents |
|---|---|
| `{DEM_PATH.name}` | Elevation (m), float32, no-data -9999 |
| `{HILLSHADE_PATH.name}` | Hillshade 0–255, light from the northwest |
| `{SLOPE_PATH.name}` | Slope (degrees) |
| `{SLUG}_dem_10m_isn93_wide.tif` | Wider copy: the ~1890 outline plus {WIDE_BUFFER_M} m, for the cross-sections |
| `{SLUG}_dem_preview.png` | Preview map |
| `{SLUG}_glacier_elevation_stats.csv` | Elevation statistics inside the newest outline |

- Projection: ISN93 / Lambert 1993 (EPSG:3057), {PIXEL:g} m pixels on the original ÍslandsDEM grid
- Extent (ISN93): x {xmin:.0f}–{xmax:.0f}, y {ymin:.0f}–{ymax:.0f} ({width} x {height} pixels)
- Download area: the ~1890 glacier outline plus {BUFFER_M} m
- A heightmap copy for the 3D viewer is in `Snaefellsjokull/Web/Model/data/`

## Glacier elevation statistics

{stats_text}

## Caveats

Each DEM pixel is the median of ArcticDEM elevations from several years (mostly 2012 to about 2020, some from 2008),
so the glacier surface is a blend of those years, not a single date. Don't use it to measure glacier thinning.
The vertical datum is not stated in the catalogue; values appear to be heights above sea level.
"""

(OUT_DIR / "README.md").write_text(readme, encoding="utf-8")
print(readme)